In [39]:
import pandas as pd
import numpy as np
from io import StringIO

# annot_df = pd.read_csv('../gen_datasets/datasets/csa_annot.csv', sep='\t')
# annot_df['sequence_len'] = annot_df['Sequence'].apply(lambda x: len(x) if isinstance(x, str) else 0)

import os
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

def hits_to_fasta(hit_df, out_dir, whitelist=None):
    
    # hit_df['win_size'] = hit_df.apply(lambda x: x['tlen'] / x['qlen'] < 1.75, axis=1)
    # hit_df = hit_df[hit_df['win_size']]

    def to_fasta(f, pid_l, seq_l):
        seq_records = [
            SeqRecord(Seq(seq), id=str(pid), description="")
            for pid, seq in zip(pid_l, seq_l)
        ]
        SeqIO.write(seq_records, f, "fasta")

    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    for group in hit_df.groupby('query'):
        query_id = group[0]
        if whitelist is None or query_id in whitelist:
            pass
            # print(f'Processing {query_id}')
            # print(query_id, group[1])
        else:
            continue
        # Filter for sequences with length within 2 standard deviations of the mean
        group_df = group[1]
        init_len = len(group_df)
        group_df = group_df[group_df['pident'] <= 95]
        m, std = np.mean(group_df['tlen']), np.std(group_df['tlen']) + 1
        group_df = group_df[(group_df['tlen'] > m - 2 * std) & (group_df['tlen'] < m + 2 * std) & (group_df['tlen'] < 1500)]

        if(len(group_df) < 5):
            # print(group_df, query_id, init_len, m, std)
            continue

        group_id = list(group_df['subject'])
        group_seq = list(group_df['tseq'])
        
        if(len(group_id) > 450):
            group_id = group_id[:450]
            group_seq = group_seq[:450]

        # print(group_df['qseq'].values[0])
        if(len(group_df['qseq'].values[0]) > 850):
            print(f'Skipping {query_id} due to long query sequence length: {len(group_df["qseq"].values[0])}')
            print(len(group_df['qseq'].values[0]))
            continue


        if(not any(query_id in s for s in group_id)):
            group_id.append(query_id)
            group_seq.append(group_df['qseq'].values[0])
        
        print('max len', max([len(seq) for seq in group_seq]), query_id, len(group_seq))
        to_fasta(f'{out_dir}/{query_id}_homologues.fasta', group_id, group_seq)

In [40]:
hit_df = pd.read_csv('csa_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
# group_sizes = hit_df.groupby('query').size()
# whitelist = ['O76290', 'P13448:P13449', 'P56740', 'Q93EK7', 'Q967M2']
hits_to_fasta(hit_df, 'uniref_msa/csa_msa')

max len 400 A0QTN8 300
max len 410 A2RJT9 305
max len 414 A4XF23 315
max len 425 A5JTM5 451
max len 983 A5JUY8 311
max len 537 E3PRJ5:E3PRJ4 308
max len 386 O04147 304
max len 444 O06644 299
max len 967 O13297 370
max len 534 O15527 277
Skipping O16025 due to long query sequence length: 1066
1066
max len 679 O28603:O28604 310
max len 326 O31156 312
max len 292 O31168 305
max len 358 O31243 348
max len 384 O31465 289
max len 150 O32727:O32728 326
max len 237 O33199 320
max len 374 O34508 304
max len 411 O34714 313
max len 512 O35543 349
max len 572 O46427 312
max len 548 O48917 298
max len 399 O50152 397
max len 378 O52942 331
max len 508 O54050:O54051 350
max len 296 O58403 328
max len 251 O58727 352
max len 216 O59413 332
max len 334 O59791 326
max len 529 O59893 287
max len 716 O64411 347
max len 259 O66188:O66187:O66186 341
max len 300 O68557 62
max len 418 O68884 307
max len 350 O69762 295
Skipping O75164 due to long query sequence length: 1064
1064
max len 301 O76290 451
max len 8

In [41]:
hit_df = pd.read_csv('llps_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
hits_to_fasta(hit_df, 'uniref_msa/llps_msa')

hit_df = pd.read_csv('elms_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
hits_to_fasta(hit_df, 'uniref_msa/elms_msa')

Skipping A0QSY1 due to long query sequence length: 899
899
Skipping A1Z813 due to long query sequence length: 1184
1184
max len 880 D0PV95 290
max len 904 D8V196 294
Skipping F1R237 due to long query sequence length: 2117
2117
max len 893 G5EBV6 33
max len 999 H0WFA5 308
Skipping J3QQ18 due to long query sequence length: 1308
1308
max len 658 O00401 293
max len 760 O00571 277
max len 1062 O13828 327
max len 355 O43561 173
max len 771 O43781 301
max len 959 O43791 256
Skipping O60500 due to long query sequence length: 1241
1241
max len 876 O60563 305
Skipping O60885 due to long query sequence length: 1362
1362
Skipping O62011 due to long query sequence length: 1054
1054
Skipping O65934 due to long query sequence length: 865
865
max len 872 O74718 315
max len 917 O94752 438
max len 421 P00873 350
max len 740 P03372 304
max len 327 P03520 30
max len 622 P03521 306
Skipping P04050 due to long query sequence length: 1733
1733
max len 804 P04147 299
max len 592 P04156 268
max len 817 P05067 

In [46]:
import os
max_len_f_l = []
for f in os.listdir('uniref_msa/csa_msa'):
    if f.endswith('.fasta'):
        sequences = SeqIO.parse(f'uniref_msa/csa_msa/{f}', 'fasta')
        seq_id = [str(seq.id) for seq in sequences]
        # max_len = max([len(seq) for seq in seqs])
        # if(max_len > 10000):
        #     print(f, seqs)
        # max_len_f_l.append(max([len(seq) for seq in seqs]))
        assert any(f.split('_')[0] in sid for sid in seq_id), f"File {f} has inconsistent sequence ID {seq_id[0]}"

In [30]:
seqs = list(SeqIO.parse('uniref_msa/csa_msa/P81453_homologues.fasta', 'fasta'))
long_seq = [seq for seq in seqs if len(seq.seq) > 10000][0]
print(long_seq.id, len(long_seq.seq))
# print([len(seq.seq) for seq in seqs])

UniRef90_UPI0033CC84C4 14478


In [27]:
max(max_len_f_l)

14478

In [ ]:
#Assemble 
!/home/andrew/anaconda3/bin/diamond blastp -d swissprot_msa/uniprot_sprot.dmnd -q ../gen_datasets/datasets/csa.fasta -o csa_matches.tsv
!/home/andrew/anaconda3/bin/diamond blastp -d swissprot_msa/uniprot_sprot.dmnd -q ../gen_datasets/datasets/llps.fasta -o llps_matches.tsv
!/home/andrew/anaconda3/bin/diamond blastp -d swissprot_msa/uniprot_sprot.dmnd -q ../gen_datasets/datasets/elms.fasta -o elms_matches.tsv